# PaliGemma, sıfırdan: gerçek ağırlıklarla doğrulama

`google/paligemma-3b-pt-224` ağırlıklarını kendi yazdığımız modele yükleyip Hugging Face ile karşılaştırıyoruz.

**Başlamadan önce**
1. **GPU:** Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU. (T4'te bf16 yavaş çalışır ama çalışır.)
2. **Hugging Face lisansı:** [google/paligemma-3b-pt-224](https://huggingface.co/google/paligemma-3b-pt-224) sayfasında lisansı kabul et. PaliGemma 2 onayı yetmiyor, bu PaliGemma 1.
3. **Hugging Face token'ı:** Colab'da sol menüde 🔑 Secrets → `HF_TOKEN` adıyla ekle, "Notebook erişimi"ni aç. Yoksa bir hücre giriş penceresi açar.
4. **Drive'a yükle:** Mac'teki `paligemma2/src/` klasörünü `Drive/paligemma2/src/` olarak kopyala. Deney görselleri için `Drive/paligemma2/images/` içine birkaç `.jpg`/`.png` koy (klasörün hemen içine, alt klasör açma) (`yz50-research/.../experiment/data/images`).

**Bellek planı (T4: 15 GB GPU, ~12 GB RAM):** bf16'da model ~6 GB. HF modelini GPU'ya alıp referansları hesaplıyoruz,
ağırlıklarını CPU'ya kopyalayıp GPU'dan siliyoruz, sonra kendi modelimizi kurup aynı ağırlıkları yüklüyoruz.
Böylece GPU'da aynı anda tek model var.

**Ne doğruluyoruz**
1. Bizim processor'ımız HF ile aynı `input_ids` ve `pixel_values`'ı veriyor mu?
2. Aynı girdi için son logit'ler HF ile yakın mı? (bf16'da birebir beklenmez, `atol≈1e-2` gevşek tolerans)
3. Greedy üretim HF `generate(do_sample=False)` ile token token aynı mı?
4. Kendi deney görsellerinde ne üretiyor?
5. KV cache (aşama 7): cache'li üretim cache'siz yolla ve HF ile aynı mı, token başına ne kadar hızlı? (Bölüm 8)

In [ ]:
import sys, time
from pathlib import Path

import torch

assert torch.cuda.is_available(), "GPU yok: Çalışma zamanı türünü T4 GPU yap"
print("GPU:", torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount("/content/drive")

# Drive'daki klasörler. Yolu kendi yerleşimine göre değiştir
DRIVE_ROOT = Path("/content/drive/MyDrive/paligemma2")
SRC_DIR = DRIVE_ROOT / "src"
IMAGES_DIR = DRIVE_ROOT / "images"

MODEL_ID = "google/paligemma-3b-pt-224"
PROMPT = "caption en"        # pt modeli görev önekleriyle eğitildi: "caption en", "ocr", "detect kedi", "answer en ..."
MAX_NEW_TOKENS = 20

assert (SRC_DIR / "paligemma.py").exists(), f"{SRC_DIR} içinde paligemma.py yok"
sys.path.insert(0, str(SRC_DIR))

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN Colab Secrets'tan alındı")
except Exception:
    from huggingface_hub import login
    login()

import transformers
print("transformers", transformers.__version__, "| torch", torch.__version__)

## 1. Referans: HF modeli

HF modelini eager attention ile (bizim yazdığımız matematikle aynı yol) bf16'da yüklüyoruz. Prefix'i iki yönlü yaptırmak için
`token_type_ids = 0` veriyoruz. Vermezsek HF görüntü + komutu da nedensel işler ve bizimle aynı şeyi hesaplamaz
(küçük rastgele modelde bu fark ~8–10 çıkmıştı, handbook'taki köprü bölümüne bak).

In [ ]:
from PIL import Image
from transformers import AutoProcessor, PaliGemmaForConditionalGeneration

# Klasörün doğrudan içindeki görseller (alt klasörlere bakılmaz). jpg, jpeg ve png, büyük/küçük harf fark etmez
image_paths = sorted(p for p in IMAGES_DIR.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
assert image_paths, f"{IMAGES_DIR} içinde görsel yok. Klasör yolunu ve içeriğini kontrol et: " + str([p.name for p in IMAGES_DIR.iterdir()][:10])
image = Image.open(image_paths[0]).convert("RGB")
print("Test görseli:", image_paths[0].name, image.size)

hf_processor = AutoProcessor.from_pretrained(MODEL_ID)
hf_inputs = hf_processor(text=PROMPT, images=image, return_tensors="pt")
hf_pixel_values_fp32 = hf_inputs["pixel_values"].clone()      # processor karşılaştırması için float32 kopya
T = hf_inputs["input_ids"].shape[1]
print("T =", T, "(256 görüntü token'ı + <bos> + komut + \\n)")
print("HF processor çıktıları:", list(hf_inputs.keys()))

In [ ]:
t0 = time.time()
hf_model = PaliGemmaForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, attn_implementation="eager"
).to("cuda").eval()
print(f"HF modeli yüklendi: {time.time() - t0:.0f} s, GPU bellek {torch.cuda.memory_allocated() / 1e9:.1f} GB")

input_ids = hf_inputs["input_ids"].to("cuda")
attention_mask = hf_inputs["attention_mask"].to("cuda")
pixel_values = hf_inputs["pixel_values"].to("cuda", torch.bfloat16)
token_type_ids = torch.zeros_like(input_ids)                    # hepsi prefix

with torch.no_grad():
    ref_logits = hf_model(input_ids=input_ids, pixel_values=pixel_values, attention_mask=attention_mask,
                          token_type_ids=token_type_ids).logits.float().cpu()      # [1, T, 257216]
    ref_out = hf_model.generate(input_ids=input_ids, pixel_values=pixel_values, attention_mask=attention_mask,
                                token_type_ids=token_type_ids, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                eos_token_id=hf_processor.tokenizer.eos_token_id,    # bizim generate'e vereceğimiz eos ile aynı
                                pad_token_id=hf_processor.tokenizer.pad_token_id)
ref_tokens = ref_out[:, T:].cpu()                               # HF komutu da geri döndürüyor
print("HF logit'leri:", tuple(ref_logits.shape))
print("HF üretimi   :", repr(hf_processor.decode(ref_tokens[0], skip_special_tokens=True)))

In [ ]:
# Ağırlıkları CPU'ya al (anahtar adları HF düzeninde, load_hf_weights bizim adlara çevirecek), GPU'yu boşalt
hf_sd = {k: v.detach().cpu() for k, v in hf_model.state_dict().items()}
print("HF state_dict anahtar sayısı:", len(hf_sd))
print("ilk 3:", list(hf_sd)[:3])
eos_token_id = hf_processor.tokenizer.eos_token_id
del hf_model
import gc; gc.collect(); torch.cuda.empty_cache()
print(f"GPU bellek: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## 2. Bizim model

Gerçek config (`PaliGemmaConfig()` varsayılanları `paligemma-3b-pt-224`'in config.json değerleri). Modeli doğrudan GPU'da ve bf16'da
kuruyoruz: önce CPU'da float32 kursak 12 GB RAM yeterli olmazdı.

`load_hf_weights` anahtarlar bire bir tutmazsa **hata verir**. Hata alırsan mesaj hangi anahtarların eşleşmediğini gösterir:
gerçek checkpoint'te adlandırma küçük rastgele modeldekinden farklıysa ilk bunu burada göreceğiz.

In [ ]:
import paligemma, generate, processing

config = paligemma.PaliGemmaConfig()
torch.set_default_dtype(torch.bfloat16)
with torch.device("cuda"):
    model = paligemma.PaliGemmaForConditionalGeneration(config)
torch.set_default_dtype(torch.float32)

paligemma.load_hf_weights(model, hf_sd)
model.eval()
del hf_sd; gc.collect()
n_params = sum(p.numel() for p in model.parameters())
print(f"Bizim model hazır: {n_params / 1e9:.2f} B parametre, GPU bellek {torch.cuda.memory_allocated() / 1e9:.1f} GB")
print("weight tying:", model.language_model.lm_head.weight is model.language_model.model.embed_tokens.weight)

## 3. Bizim processor HF ile aynı mı?

In [ ]:
our_processor = processing.PaliGemmaProcessor(
    tokenizer=hf_processor.tokenizer, num_image_tokens=config.text_config.num_image_tokens,
    image_size=config.vision_config.image_size,
)
ours = our_processor(text=[PROMPT], images=[image])

ids_equal = torch.equal(ours["input_ids"], hf_inputs["input_ids"])
pix_diff = (ours["pixel_values"].float() - hf_pixel_values_fp32).abs().max().item()
print(f"input_ids aynı mı: {ids_equal} ({tuple(ours['input_ids'].shape)})")
print(f"pixel_values max fark: {pix_diff:.2e}")

## 4. Logit karşılaştırması

bf16'da iki implementasyon aynı matematiği farklı işlem sırasıyla yapınca küçük farklar çıkar, 45 katmandan (27 SigLIP + 18 Gemma)
sonra birikir. Bu yüzden birebir değil, şunlara bakıyoruz: en büyük/ortalama fark, her konumda `argmax` aynı mı,
son konumda top-5 aynı mı. Asıl ölçüt bir sonraki bölüm: üretilen token'lar aynı mı.

In [ ]:
with torch.no_grad():
    logits = model(input_ids, pixel_values, attention_mask).float().cpu()    # [1, T, 257216]

diff = (logits - ref_logits).abs()
print(f"şekil: bizim {tuple(logits.shape)}, HF {tuple(ref_logits.shape)}")
print(f"max fark {diff.max():.3f}, ortalama fark {diff.mean():.5f}, logit ölçeği (|HF| ortalama) {ref_logits.abs().mean():.2f}")
print(f"allclose(atol=1e-2): {torch.allclose(logits, ref_logits, atol=1e-2)}")

agree = (logits.argmax(-1) == ref_logits.argmax(-1)).float()
print(f"argmax aynı olan konum oranı: {agree.mean():.4f}  (son konum: {bool(agree[0, -1])})")
top5_ours, top5_hf = logits[0, -1].topk(5).indices.tolist(), ref_logits[0, -1].topk(5).indices.tolist()
print("son konum top-5 bizim:", top5_ours)
print("son konum top-5 HF   :", top5_hf)

## 5. Greedy üretim: HF ile token token

Bizim `generate` varsayılan olarak KV cache kullanıyor (aşama 7). Cache'siz yolla karşılaştırma ve süre ölçümü Bölüm 8'de.
bf16'da iki token'ın logit'i çok yakınsa sıralama değişebilir. Böyle bir sapma çıkarsa ilk farklı konumu ve iki adayı yazdırıyoruz.

In [ ]:
t0 = time.time()
our_tokens = generate.generate(model, input_ids, pixel_values, attention_mask, MAX_NEW_TOKENS, eos_token_id).cpu()
print(f"üretim süresi: {time.time() - t0:.1f} s ({our_tokens.shape[1]} token, KV cache'li)")

n = min(our_tokens.shape[1], ref_tokens.shape[1])
same = torch.equal(our_tokens[:, :n], ref_tokens[:, :n]) and our_tokens.shape[1] == ref_tokens.shape[1]
print("bizim:", repr(hf_processor.decode(our_tokens[0], skip_special_tokens=True)))
print("HF   :", repr(hf_processor.decode(ref_tokens[0], skip_special_tokens=True)))
print("token token aynı:", same)
if not same:
    mismatch = (our_tokens[0, :n] != ref_tokens[0, :n]).nonzero()
    if len(mismatch):
        i = mismatch[0].item()
        print(f"ilk fark {i}. token'da: bizim {our_tokens[0, i].item()} ({hf_processor.decode(our_tokens[0, i])!r}), "
              f"HF {ref_tokens[0, i].item()} ({hf_processor.decode(ref_tokens[0, i])!r})")

## 6. Kendi deney görsellerinde

Görseller Türkçe tabela ve fiş fotoğrafları. `pt` modeli görev önekleriyle çalışır: `ocr` metni okur, `caption en` İngilizce açıklar,
`answer en <soru>` soruyu cevaplar. Üretim KV cache'li, adım başına süre kısa.

In [ ]:
from IPython.display import display

PROMPTS = ["caption en", "ocr"]
for path in image_paths[:3]:
    img = Image.open(path).convert("RGB")
    display(img.resize((img.width * 240 // img.height, 240)))
    for prompt in PROMPTS:
        enc = our_processor(text=[prompt], images=[img])
        t0 = time.time()
        out = generate.generate(model, enc["input_ids"].to("cuda"), enc["pixel_values"].to("cuda", torch.bfloat16),
                                enc["attention_mask"].to("cuda"), MAX_NEW_TOKENS, eos_token_id).cpu()
        print(f"[{path.name}] {prompt!r} -> {hf_processor.decode(out[0], skip_special_tokens=True)!r}  ({time.time() - t0:.0f} s)")

## 7. Teşhis: logit farkı bf16 gürültüsü mü, yoksa bir sapma mı?

Bölüm 4'te ortalama fark 0.19, max fark 3.9 çıktı ve `allclose(atol=1e-2)` tutmadı. bf16'da bir miktar fark beklenir, ama
sayının büyüklüğü tek başına bir hatayı dışlamıyor. Üç adımda ayırt ediyoruz:

- **7a. Farkın dağılımı** (HF'yi yeniden yüklemeden): fark nerede yoğun, argmax'ın uyuşmadığı yerlerde HF zaten kararsız mıydı?
- **7b. Katman katman göreli hata** (HF yeniden yüklenir, ikisi birden GPU'da): hata yumuşak mı büyüyor, yoksa bir yerde zıplıyor mu?
- **7c. fp32 testi** (CPU'da): aynı gerçek ağırlıklarla SigLIP'in tamamı ve birkaç Gemma katmanı **float32**'de HF ile aynı mı?
  Aynıysa kod doğru, kalan fark bf16 yuvarlamasıdır.

In [ ]:
# 7a. Farkın dağılımı. Yeni model yüklemiyor, Bölüm 4'teki `logits` ve `ref_logits`'i kullanıyor
n_img = config.text_config.num_image_tokens
diff = (logits - ref_logits).abs()                                   # [1, T, V]
agree = (logits.argmax(-1) == ref_logits.argmax(-1))[0]              # [T]
top2 = ref_logits[0].topk(2, dim=-1).values
margin = top2[:, 0] - top2[:, 1]                                     # HF'nin 1. ve 2. adayı arasındaki fark: küçükse HF de emin değil

print(f"{'bölge':20s} {'konum':>6s} {'ort |fark|':>11s} {'argmax aynı':>12s} {'HF top1-top2':>13s}")
for name, sl in [("görüntü konumları", slice(0, n_img)), ("<bos> + komut + \\n", slice(n_img, T))]:
    print(f"{name:20s} {sl.stop - sl.start:6d} {diff[0, sl].mean():11.3f} {agree[sl].float().mean():12.3f} {margin[sl].mean():13.2f}")

dis = ~agree
print(f"\nargmax'ın uyuşmadığı {int(dis.sum())} konumun {int(dis[:n_img].sum())} tanesi görüntü konumu")
if dis.any() and agree.any():
    print(f"HF'nin top1-top2 farkı: uyuşmayanlarda ortalama {margin[dis].mean():.2f}, uyuşanlarda {margin[agree].mean():.2f}")
    print("(uyuşmayanlarda belirgin biçimde küçükse: farklar HF'nin zaten kararsız olduğu konumlarda, bf16 gürültüsü sıralamayı çeviriyor)")
per_pos = diff[0].mean(-1)
print("\nen büyük ortalama farklı 5 konum:", [(int(i), round(float(per_pos[i]), 2), "görüntü" if i < n_img else "metin") for i in per_pos.topk(min(5, T)).indices])
print(f"son konum: ort |fark| {per_pos[-1]:.3f}, HF top1-top2 {margin[-1]:.2f}, argmax aynı: {bool(agree[-1])}")

### 7b. Katman katman göreli hata

HF'yi bir kez daha yüklüyoruz (önbellekten, ilk seferden çok daha hızlı). İki model aynı girdiyle çalışırken her bloğun çıkışını `forward hook` ile kaydedip
**göreli hata** hesaplıyoruz: `‖bizim − HF‖ / ‖HF‖`. Aynı `pixel_values` kullanıldığı için Bölüm 3'teki 1 seviyelik piksel farkı bu testi etkilemiyor.

Nasıl okunur: bf16'da tek işlemde göreli yuvarlama ~4e-3. 45 katmandan sonra %1–5 göreli hata olağan. Hata **yumuşakça** büyüyorsa gürültü, bir katmanda
**birden zıplıyorsa** (satırda `<<< sıçrama`) o katmanda bir fark var demektir.

In [ ]:
# 7b. HF'yi tekrar yükle, iki modelin ara çıktılarını hook'larla yakala
hf2 = PaliGemmaForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, attn_implementation="eager"
).to("cuda").eval()
device = next(model.parameters()).device
print(f"GPU bellek (iki model): {torch.cuda.memory_allocated() / 1e9:.1f} GB" if device.type == "cuda" else "cpu")


def first_attr(obj, *paths):
    # transformers sürümüne göre adlar değişiyor (5.x: model.vision_tower, 4.x: vision_tower)
    for path in paths:
        o = obj
        try:
            for part in path.split("."):
                o = getattr(o, part)
            return o
        except AttributeError:
            continue
    raise AttributeError(f"hiçbiri yok: {paths}")


def as_tensor(x):
    if isinstance(x, tuple):
        x = x[0]
    if hasattr(x, "last_hidden_state"):
        x = x.last_hidden_state
    return x


hf_vt = first_attr(hf2, "model.vision_tower", "vision_tower")
hf_vt_in = getattr(hf_vt, "vision_model", hf_vt)
hf_proj = first_attr(hf2, "model.multi_modal_projector", "multi_modal_projector")
hf_lm = first_attr(hf2, "model.language_model", "language_model.model")
our_vt = model.vision_tower
our_vt_in = our_vt.vision_model
our_lm = model.language_model.model

# RoPE frekansları iki tarafta da float32 ve aynı olmalı. HF bunları bf16'ya yuvarlamışsa (bir .to(bfloat16) ile olabilir)
# her Gemma katmanında küçük bir fark çıkar
hf_inv, our_inv = first_attr(hf_lm, "rotary_emb").inv_freq, our_lm.rotary_emb.inv_freq
print(f"RoPE inv_freq: HF {hf_inv.dtype}, bizim {our_inv.dtype}, max fark {(hf_inv.float().cpu() - our_inv.float().cpu()).abs().max().item():.2e}")

pairs = [("SigLIP embeddings", hf_vt_in.embeddings, our_vt_in.embeddings)]
pairs += [(f"SigLIP katman {i:2d}", h, o) for i, (h, o) in enumerate(zip(hf_vt_in.encoder.layers, our_vt_in.encoder.layers))]
pairs += [("SigLIP çıkışı", hf_vt, our_vt), ("projektör", hf_proj, model.multi_modal_projector)]
pairs += [(f"Gemma katman {i:2d}", h, o) for i, (h, o) in enumerate(zip(hf_lm.layers, our_lm.layers))]
pairs += [("Gemma son norm", hf_lm.norm, our_lm.norm)]

hf_acts, our_acts, handles = {}, {}, []


def save_output(store, name):
    # Forward hook None döndürmeli: bir değer döndürürse modülün çıktısını O DEĞERLE değiştirir (setdefault döndürürdü)
    def hook(module, args, output):
        store.setdefault(name, as_tensor(output).detach().float().cpu())
    return hook


for name, h_mod, o_mod in pairs:
    handles.append(h_mod.register_forward_hook(save_output(hf_acts, name)))
    handles.append(o_mod.register_forward_hook(save_output(our_acts, name)))


def grab_first_input(store, key):
    # Gemma'nın 1. katmanına giren tensör (7c'de fp32 testinin girdisi olacak)
    def hook(module, args, kwargs):
        store.setdefault(key, (args[0] if args else kwargs["hidden_states"]).detach().float().cpu())
    return hook


handles.append(hf_lm.layers[0].register_forward_pre_hook(grab_first_input(hf_acts, "gemma_in0"), with_kwargs=True))

with torch.no_grad():
    hf2(input_ids=input_ids, pixel_values=pixel_values, attention_mask=attention_mask,
        token_type_ids=torch.zeros_like(input_ids))
    model(input_ids, pixel_values, attention_mask)
for h in handles:
    h.remove()

print(f"{'blok':22s} {'göreli hata':>12s} {'max |fark|':>11s}")
prev = None
for name, _, _ in pairs:
    a, b = our_acts[name], hf_acts[name]
    if a.shape != b.shape:
        print(f"{name:22s} ŞEKİL FARKLI: bizim {tuple(a.shape)}, HF {tuple(b.shape)}")
        continue
    rel = ((a - b).norm() / b.norm()).item()
    flag = "  <<< sıçrama" if prev is not None and rel > 3 * prev and rel > 2e-3 else ""
    print(f"{name:22s} {rel:12.2e} {(a - b).abs().max().item():11.3f}{flag}")
    prev = rel

### 7c. fp32 testi: aynı gerçek ağırlıklar, bf16 yuvarlaması olmadan

bf16 ağırlıklar float32'ye çevrilince değerleri aynen korunur. SigLIP'in tamamını (0.4 B parametre) ve Gemma'nın birkaç katmanını **CPU'da float32** ile
hem HF hem bizim kodla çalıştırıp karşılaştırıyoruz. Gemma katmanlarının girdisi olarak 7b'de yakalanan HF ara çıktıları kullanılıyor. Fark ~1e-5 civarındaysa
kodumuz HF ile aynı fonksiyonu hesaplıyor ve Bölüm 4'teki fark yalnızca bf16'dan geliyor.

In [ ]:
# 7c. fp32 testi (CPU). Bir test hata verirse diğerleri yine de çalışır
import copy
import paligemma


def fp32_cpu_copy(module):
    # GPU'da fazladan kopya açmamak için önce CPU'ya taşıyıp orada çoğaltıyoruz, sonra geri koyuyoruz
    module.to("cpu")
    copied = copy.deepcopy(module).float().eval()
    module.to(device)
    return copied


def report(name, a, b):
    a, b = as_tensor(a).float(), as_tensor(b).float()
    rel = ((a - b).norm() / b.norm()).item()
    print(f"[{'OK ' if rel < 1e-4 else '???'}] {name}: göreli hata {rel:.2e}, max |fark| {(a - b).abs().max().item():.2e}")


with torch.no_grad():
    try:
        px = pixel_values.float().cpu()
        report("SigLIP (27 katman, fp32)", fp32_cpu_copy(our_vt)(px), fp32_cpu_copy(hf_vt)(px))
    except Exception as e:
        print(f"SigLIP fp32 testi çalışmadı: {type(e).__name__}: {e}")

    n_layers = len(our_lm.layers)
    hf_rot, our_rot = copy.deepcopy(first_attr(hf_lm, "rotary_emb")).cpu(), copy.deepcopy(our_lm.rotary_emb).cpu()
    pos = torch.arange(1, T + 1).unsqueeze(0)                          # PaliGemma konumları 1'den başlıyor
    mask = paligemma.make_prefix_lm_mask(T, T, torch.float32, torch.device("cpu"))
    for i in sorted({0, n_layers // 2, n_layers - 1}):
        try:
            x = (hf_acts["gemma_in0"] if i == 0 else hf_acts[f"Gemma katman {i - 1:2d}"]).float()   # HF'nin kendi ara çıktısı
            cos_h, sin_h = hf_rot(x, pos)
            cos_o, sin_o = our_rot(x, pos)
            out_hf = fp32_cpu_copy(hf_lm.layers[i])(x, attention_mask=mask, position_embeddings=(cos_h, sin_h))
            out_ours = fp32_cpu_copy(our_lm.layers[i])(x, (cos_o, sin_o), mask)
            report(f"Gemma katman {i:2d} (fp32)", out_ours, out_hf)
        except Exception as e:
            print(f"Gemma katman {i} fp32 testi çalışmadı: {type(e).__name__}: {e}")

### 7 sonu: nasıl yorumlarım?

| Gördüğün | Anlamı |
|---|---|
| 7a: uyuşmayan konumlar çoğunlukla görüntü konumları ve HF'nin top1-top2 farkı orada küçük | Eğitimde öğretilmemiş, kararsız konumlarda bf16 sıralamayı çeviriyor. Zararsız |
| 7b: göreli hata yumuşak büyüyor, sıçrama yok | bf16 birikimi |
| 7b: SigLIP ve projektör ~0, **Gemma katman 0'da sıçrama**, 7c hepsi `OK` | Fark Gemma'ya girişte: `PaliGemma._merge_input_ids_with_image_features` görüntü özelliklerini `÷√d` sonra `×√d` yapıyor, bf16'da iki yuvarlama. HF (5.x) görüntüyü hiç ölçeklemiyor, metni tabloda `×√d` yapıyor. Yerelde küçük modelde (d=48) bu yolun fark yarattığını, HF gibi birleştirince farkın sıfırlandığını gördük |
| 7b: başka bir blokta sıçrama | O bloğun girdisini/çıktısını incele, sebebi orada |
| 7c: hepsi `OK` (~1e-5 ve altı) | Kod gerçek ağırlıklarla HF ile aynı fonksiyonu hesaplıyor. Kalan fark yalnızca bf16 |
| 7c: bir blok `???` | O blokta gerçek bir fark var, bf16 değil |

In [ ]:
# Temizlik: HF modelini GPU'dan at
# `pairs` HF modüllerine referans tutuyor, onu da silmezsek bellek boşalmaz
del pairs, handles, hf2, hf_vt, hf_vt_in, hf_proj, hf_lm, hf_acts, our_acts, hf_rot, our_rot
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU bellek: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## 8. KV cache (aşama 7)

Aynı girdiyle iki yol:
- `use_cache=False` (aşama 6): her adımda görüntü + komut + şimdiye kadarki cevap ve SigLIP baştan hesaplanıyor.
- `use_cache=True`: görüntü + komut bir kez işleniyor (prefill), sonra her adımda modele sadece son token giriyor.

**Doğruluk:** iki yol birbiriyle ve Bölüm 1'deki HF üretimiyle token token aynı olmalı. bf16'da cache'li ve cache'siz yolun logit'leri
birebir aynı değil (farklı şekilli matmul'lar toplamayı farklı sırayla yapıyor). Çok yakın iki aday varsa uzun üretimde nadiren bir token
ayrışabilir. Ayrışırsa ilk farklı konumu yazdırıyoruz.

**Süre:** GPU asenkron çalıştığı için ölçümden önce `torch.cuda.synchronize()`. Ayrıca eos'suz (`eos_token_id=-1`) 64 token üretip
cache'siz yolun dizi uzadıkça nasıl yavaşladığına bakıyoruz. Yerelde (M1, gerçek boyutlu ama az katmanlı model) cache'li adım ~44 ms'de
sabit kaldı, cache'siz adım 353 ms'den 472 ms'ye çıktı (`checks/bench_kv_cache.py`).

In [ ]:
def timed_generate(use_cache, max_new_tokens, eos):
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    out = generate.generate(model, input_ids, pixel_values, attention_mask, max_new_tokens, eos, use_cache=use_cache).cpu()
    torch.cuda.synchronize()
    return out, time.perf_counter() - t0


def first_mismatch(a, b):
    n = min(a.shape[1], b.shape[1])
    diff = (a[0, :n] != b[0, :n]).nonzero()
    return diff[0].item() if len(diff) else (None if a.shape[1] == b.shape[1] else n)


timed_generate(True, 2, eos_token_id)                  # ısınma: ilk çağrıda CUDA kernel'leri yükleniyor

# 1) Doğruluk: Bölüm 5'teki ayarlarla (eos'ta duruyor)
cached, t_cached = timed_generate(True, MAX_NEW_TOKENS, eos_token_id)
uncached, t_uncached = timed_generate(False, MAX_NEW_TOKENS, eos_token_id)
print("cache    :", repr(hf_processor.decode(cached[0], skip_special_tokens=True)))
print("cache yok:", repr(hf_processor.decode(uncached[0], skip_special_tokens=True)))
print("HF       :", repr(hf_processor.decode(ref_tokens[0], skip_special_tokens=True)))
for name, other in [("cache yok", uncached), ("HF", ref_tokens)]:
    i = first_mismatch(cached, other)
    print(f"cache = {name}: {i is None}" + ("" if i is None else f" (ilk fark {i}. token'da)"))

# 2) Süre: eos'suz 64 token, iki yol da tam olarak 64 adım
N = 64
long_cached, t_long_cached = timed_generate(True, N, -1)
long_uncached, t_long_uncached = timed_generate(False, N, -1)
print(f"\n{N} token, eos yok:")
print(f"   cache yok: {t_long_uncached:6.2f} s, {t_long_uncached / N * 1e3:6.1f} ms/token")
print(f"   cache    : {t_long_cached:6.2f} s, {t_long_cached / N * 1e3:6.1f} ms/token")
i = first_mismatch(long_cached, long_uncached)
print(f"   hızlanma {t_long_uncached / t_long_cached:.1f}x, token'lar aynı: {i is None}" + ("" if i is None else f" (ilk fark {i}. token'da)"))

In [ ]:
# Adım adım süre: cache'siz adım dizi uzunluğuyla büyüyor, cache'li adım neredeyse sabit.
# Prefill (ilk adım) iki yolda da aynı iş, sonraki adımlar ayrışıyor
from paligemma import KVCache

with torch.no_grad():
    for use_cache in (False, True):
        cache = KVCache() if use_cache else None
        ids, times = input_ids, []
        for step in range(N):
            torch.cuda.synchronize(); t0 = time.perf_counter()
            if step == 0 or not use_cache:
                logits = model(ids, pixel_values, torch.ones_like(ids), prefix_len=T, kv_cache=cache)
            else:
                logits = model(ids[:, -1:], None, torch.ones_like(ids[:, -1:]), kv_cache=cache)
            torch.cuda.synchronize(); times.append(time.perf_counter() - t0)
            ids = torch.cat([ids, logits[:, -1:].argmax(-1)], dim=1)
        picks = [0, 1, N // 4, N // 2, N - 1]
        print(f"{'cache' if use_cache else 'cache yok':>9}: " + ", ".join(f"adım {s}: {times[s] * 1e3:.0f} ms" for s in picks))

## Sonuç

- Bölüm 3: processor aynı çıktıyı veriyorsa görüntü ve metin hazırlığı doğru.
- Bölüm 4: argmax oranı ~1 ve ortalama fark küçükse model HF ile aynı fonksiyonu hesaplıyor.
- Bölüm 5: token token aynıysa aşama 6'nın doğrulaması tamam. Küçük bir sapma bf16'dan geliyorsa aday logit'lerinin yakınlığını incele.
- Bölüm 7: logit farkının kaynağı. 7c'deki fp32 testleri `OK` ise kod doğru, fark bf16 yuvarlamasından.
- Bölüm 8: cache'li, cache'siz ve HF üretimi token token aynıysa aşama 7'nin doğrulaması tamam. ms/token farkı KV cache'in kazancı.
- Sıradaki: Gemma 2 farkları → PaliGemma 2 (aşama 8).